# Deep Agent with State Backend

Demonstrates how to use `create_deep_agent()` with a **StateBackend** for in-memory file operations.

**Features covered:**
- Pre-populating a virtual filesystem with `create_file_data()`
- Using `StateBackend` as the backend callable
- Streaming agent responses with `stream_mode=["values"]`
- Setting the `name` parameter for better trace identification

**Prerequisites:**
- `deepagents`, `langchain-openai`, `python-dotenv` packages
- `OPENAI_API_KEY` environment variable (loaded from `~/.env/orchestra/.env.backend`)

In [ ]:
!uv pip install -q deepagents langchain-openai python-dotenv

## Environment Setup

Load API keys from the standard Orchestra env file.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from standard location
# ~/.env/orchestra/.env.backend
env_path = Path.home() / ".env" / "orchestra" / ".env.backend"
load_dotenv(env_path)

## Create and Run the Agent

Pre-populate a virtual filesystem, create a `StateBackend`-powered agent, and stream its response.

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends.utils import create_file_data
from deepagents.backends.state import StateBackend
from langchain.chat_models import init_chat_model

# Pre-configure files
initial_files = {
    "/project/README.md": create_file_data("# My Project\n\nInitial documentation."),
    "/project/src/app.py": create_file_data("def main():\n    print('Hello!')")
}

# Create agent using the public API — pass StateBackend as a callable
model = init_chat_model(model="openai:gpt-4.1-mini")
agent = create_deep_agent(
    backend=StateBackend,
    model=model,
    name="state-backend-demo",
)
input = {
    "messages": [{"role": "user", "content": "List files in project directory."}],
    "files": initial_files,
}
for mode, chunk in agent.stream(
    input,
    stream_mode=["values"],
    config={"configurable": {"thread_id": "openai"}},
):
    if "messages" in chunk:
        chunk["messages"][-1].pretty_print()